[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/data-engineering-from-scratch-to-advanced/blob/main/notebooks/topics/04_sql-advanced.ipynb)

*Data Engineering from Scratch to Advanced · Notebook 4 of 21*

# Advanced SQL

The SQL that separates beginners from professionals: CTEs, every window function, top-N per group, running totals, sessionisation, gaps and islands, deduplication, retention, funnels and recursive queries.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** A window function looks at neighbouring rows without squashing them. Once that clicks, hard SQL becomes pattern matching.

**What interviewers really test here:** Top N per group, latest row per key, running totals, sessions and gaps and islands.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. CTEs: write SQL you can read *(Basics)*
- [ ] 2. Window functions: the big idea *(Core)*
- [ ] 3. Running totals, moving averages and LAG *(Core)*
- [ ] 4. Sessions, streaks and gaps *(Advanced)*
- [ ] 5. Funnels, retention, pivots and hierarchies *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline
import math
import numpy as np
import matplotlib.pyplot as plt
PLAIN = True

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Picking the tool

| Question sounds like | Reach for |
| --- | --- |
| "latest for each…", "keep one per…" | ROW\_NUMBER = 1 |
| "top N per…" | RANK / DENSE\_RANK in a CTE |
| "running…", "cumulative…" | SUM() OVER (ORDER BY) |
| "compared to previous…" | LAG |
| "moving average" | AVG() OVER (ROWS n PRECEDING) |
| "consecutive…" | value − ROW\_NUMBER |
| "sessions" | LAG, then a running SUM of new-session flags |
| "share of total" | x / SUM(x) OVER (PARTITION BY g) |
| "hierarchy", "reports to" | recursive CTE |

---
# Part 2: Lessons, step by step

## Lesson 1: CTEs: write SQL you can read

*Level: Basics*

Long nested queries read inside out. A **CTE** (common table expression) names each step, so a query reads top to bottom like a recipe:



```python
WITH totals AS (                 -- step 1: revenue per region
    SELECT region, SUM(amount) AS t FROM sales GROUP BY region
),
big AS (                         -- step 2: only the big ones
    SELECT * FROM totals WHERE t > 500
)
SELECT * FROM big;
```



Regions N and S total 600 and 500, so only N remains.



Try these in the [SQL IDE](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/ide/sql.html): choose the Analytics database, or open the Practice tab.



Professional SQL is mostly chains of CTEs: import the data, clean it, join it, aggregate it. Each step can be checked on its own by selecting from it, which makes debugging far easier.

**✏️ Your turn:** you now have what you need for [Q1](#q1). Try one now, or carry on and do them in Part 3.

## Lesson 2: Window functions: the big idea

*Level: Core*

GROUP BY squashes each group into one row. A **window function** computes across related rows, but **every row stays**.



```python
function() OVER (PARTITION BY region ORDER BY amount DESC)
                 └── separate groups ┘ └── order inside ─┘
```



**The three rankings differ only on ties.** Amounts 300, 300, 200, 100:



| Amount | ROW\_NUMBER | RANK | DENSE\_RANK |
| --- | --- | --- | --- |
| 300 | 1 | 1 | 1 |
| 300 | 2 | 1 | 1 |
| 200 | 3 | **3** | **2** |
| 100 | 4 | 4 | 3 |



- **ROW\_NUMBER:** always unique. Use it to keep exactly one row per group.
- **RANK:** ties share a rank, then there's a gap.
- **DENSE\_RANK:** no gap. Use it for "top 2 distinct values".



**You can't filter on a window function in WHERE,** because windows run after WHERE. Wrap it in a CTE, as in the top-N pattern you'll write hundreds of times:



```python
WITH r AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY region ORDER BY amount DESC) AS rn
  FROM sales)
SELECT * FROM r WHERE rn = 1;
```



**The most used pattern in data engineering is "latest row per key".** Sources resend and update records, and you keep the newest. Order by updated time descending, and add a tiebreaker so reruns give identical results.



The demo prints all three rankings. Change the data and predict the output first.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [2]:
import sqlite3
DEMO = """
CREATE TABLE sales (name TEXT, region TEXT, amount INT);
INSERT INTO sales VALUES ('a','N',100),('b','N',300),('c','N',300),('d','N',200),('e','S',50);
"""
def sql(query):
    con = sqlite3.connect(":memory:")
    con.executescript(DEMO)
    return con.execute(query).fetchall()


def rank_demo():
    for row in sql("""SELECT name, amount, ROW_NUMBER() OVER w, RANK() OVER w, DENSE_RANK() OVER w
                      FROM sales WHERE region = 'N' WINDOW w AS (ORDER BY amount DESC)"""):
        print(row)


rank_demo()

('b', 300, 1, 1, 1)
('c', 300, 2, 1, 1)
('d', 200, 3, 3, 2)
('a', 100, 4, 4, 3)


**✏️ Your turn:** you now have what you need for [Q2](#q2), [Q4](#q4), [Q5](#q5), [Q10](#q10), [Q3](#q3), [Q16](#q16). Try one now, or carry on and do them in Part 3.

## Lesson 3: Running totals, moving averages and LAG

*Level: Core*

**Running total:** `SUM(x) OVER (ORDER BY day)`. For 100, 300, 200 it gives 100, 400, 600.



**Frames** choose which rows the window includes:



| Frame | Meaning |
| --- | --- |
| `ROWS UNBOUNDED PRECEDING` | from the start to here |
| `ROWS 1 PRECEDING` | this row and the one before |
| `ROWS BETWEEN 3 PRECEDING AND CURRENT ROW` | a 4-row window |



For 50, 50, 400, a 2-row average gives 50, 50, 225. The first row only has itself.



**A trap:** with ORDER BY and no frame, the default is `RANGE`, and tied rows are treated as one. Rows with equal keys get the same running total. Use `ROWS` and a unique ordering when you want row by row.



**LAG and LEAD** read the previous and next row. For 100, 300, 200, the change is NULL, +200, −100. `FIRST_VALUE` gets the first in the window.



**Share of total without collapsing:** `amount / SUM(amount) OVER (PARTITION BY region)`. Region N of 100, 300, 200 gives 16.7%, 50%, 33.3%.

**✏️ Your turn:** you now have what you need for [Q6](#q6), [Q7](#q7), [Q8](#q8), [Q9](#q9), [Q15](#q15). Try one now, or carry on and do them in Part 3.

## Lesson 4: Sessions, streaks and gaps

*Level: Advanced*

These look hard, and they're asked a lot. Learn the two tricks and they become routine.



**Trick 1: sessionisation.** A new session starts after a gap of over 30 minutes (1800 seconds). Mark where gaps occur, then take a running sum of those marks.



| ts | gap | new? | running sum + 1 = session |
| --- | --- | --- | --- |
| 100 | none | 0 | 1 |
| 160 | 60 | 0 | 1 |
| 2000 | 1840 | **1** | 2 |
| 2050 | 50 | 0 | 2 |



**Trick 2: gaps and islands.** Find consecutive runs by subtracting a row number. Logins on days 1, 2, 3, 5, 6:



| Day | Row number | Day − row |
| --- | --- | --- |
| 1 | 1 | **0** |
| 2 | 2 | **0** |
| 3 | 3 | **0** |
| 5 | 4 | **1** |
| 6 | 5 | **1** |



Within a run, both rise by 1, so the difference stays constant. Group by it and count: runs of 3 and 2, so the longest is **3**. The same trick finds free seat ranges: 3–5 and 7–8.



**Date spines.** A report with no sales on a day should show 0, not skip it. Generate all dates, then LEFT JOIN the facts.

**✏️ Your turn:** you now have what you need for [Q17](#q17), [Q18](#q18), [Q11](#q11), [Q14](#q14). Try one now, or carry on and do them in Part 3.

## Lesson 5: Funnels, retention, pivots and hierarchies

*Level: Advanced*

**Funnel:** how many users reached each step? Make one row per user with a flag for each step, and then sum.



```python
SELECT user_id, MAX(page = 'home') AS h, MAX(page = 'cart') AS c ...
```



**Cohort retention.** Group users by signup week, and count who's active N weeks later. Join activity to signups, and compute `activity_week − signup_week`. Cohort 0 has 3 users: 3 active in week 0, 1 in week 1, 1 in week 2. Count *distinct* users, since one person can have many events.



**Pivot:** `SUM(CASE WHEN region = 'N' THEN amount ELSE 0 END)` turns rows into columns.



**Recursive CTE:** walks trees and graphs. A starting row, then a step that joins back to itself, until no new rows appear:



```python
WITH RECURSIVE t(id, name, d) AS (
    SELECT id, name, 0 FROM tree WHERE parent IS NULL
    UNION ALL
    SELECT c.id, c.name, t.d + 1 FROM tree c JOIN t ON c.parent = t.id)
SELECT * FROM t;
```



Beware of cycles in the data, which would make it loop forever. Add a depth limit.

**✏️ Your turn:** you now have what you need for [Q12](#q12), [Q19](#q19), [Q13](#q13), [Q20](#q20). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

**▶ Run this cell before the questions.** It sets up the data and tools they share.

In [3]:
import sqlite3

SCHEMA = """
CREATE TABLE events (user_id INT, ts INT, page TEXT);
INSERT INTO events VALUES (1,100,'home'),(1,160,'cart'),(1,2000,'home'),(1,2050,'buy'),(2,300,'home'),(2,4000,'home'),(3,500,'cart');
CREATE TABLE sales (id INT, region TEXT, rep TEXT, day TEXT, amount INT);
INSERT INTO sales VALUES (1,'N','a','2026-01-01',100),(2,'N','b','2026-01-02',300),(3,'N','a','2026-01-03',200),
 (4,'S','c','2026-01-01',50),(5,'S','d','2026-01-02',50),(6,'S','c','2026-01-04',400);
CREATE TABLE customer_updates (id INT, customer INT, email TEXT, updated INT);
INSERT INTO customer_updates VALUES (1,1,'a@x',10),(2,1,'a2@x',20),(3,2,'b@x',15),(4,2,'b@x',15),(5,3,'c@x',5);
CREATE TABLE logins (user_id INT, day INT);
INSERT INTO logins VALUES (1,1),(1,2),(1,3),(1,5),(1,6),(2,2),(2,4),(3,1),(3,2),(3,8);
CREATE TABLE tree (id INT, parent INT, name TEXT);
INSERT INTO tree VALUES (1,NULL,'all'),(2,1,'food'),(3,1,'home'),(4,2,'tea'),(5,4,'green');
CREATE TABLE signups (user_id INT, week INT);
INSERT INTO signups VALUES (1,0),(2,0),(3,0),(4,1),(5,1);
CREATE TABLE activity (user_id INT, week INT);
INSERT INTO activity VALUES (1,0),(1,1),(2,0),(2,2),(3,0),(4,1),(4,2),(5,1);
CREATE TABLE seats (seat INT, taken INT);
INSERT INTO seats VALUES (1,1),(2,1),(3,0),(4,0),(5,0),(6,1),(7,0),(8,0);
"""

def run_sql(query):
    """Run a query on a fresh database and return the rows as lists."""
    con = sqlite3.connect(":memory:")
    con.executescript(SCHEMA)
    rows = con.execute(query).fetchall()
    con.close()
    return [list(r) for r in rows]


<a id="q1"></a>
### Q1. Readable Steps With a CTE

🟢 Warm-up · *WITH*

Using a CTE of each region's total, return regions whose total exceeds 500.

**Examples**

- `` → `[['N', 600]]`

In [ ]:
def big_regions():
    """
    Using a CTE of each region's total, return regions whose total exceeds 500.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(big_regions, [
    ((), [['N', 600]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A nested subquery. It works, but reads inside out.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Name the step with WITH, so it reads top to bottom.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A nested subquery. It works, but reads inside out.

```python
def big_regions():
    return run_sql("SELECT region, t FROM (SELECT region, SUM(amount) AS t FROM sales GROUP BY region) WHERE t > 500")
```

**Cost:** O(n) time, O(r) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Name the step with WITH, so it reads top to bottom.

```python
def big_regions():
    return run_sql("""WITH totals AS (SELECT region, SUM(amount) AS t FROM sales GROUP BY region)
        SELECT region, t FROM totals WHERE t > 500""")
```

**Cost:** O(n) time, O(r) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A nested subquery. It works, but reads inside out. That costs O(n) time and O(r) space."

"I can do better: Name the step with WITH, so it reads top to bottom. That's O(n) time and O(r) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/sql-advanced/readable-steps-with-a-cte)

<a id="q2"></a>
### Q2. Number the Rows

🟢 Warm-up · *ROW\_NUMBER*

Return id and its order by amount (highest first) within each region, ordered by region, then that number. Ties by id.

**Examples**

- `` → `[['N', 2, 1], ['N', 3, 2], ['N', 1, 3], ['S', 6, 1], ['S', 4, 2], ['S', 5, 3]]`

In [ ]:
def numbered():
    """
    Return id and its order by amount (highest first) within each region,
    ordered by region, then that number. Ties by id.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(numbered, [
    ((), [['N', 2, 1], ['N', 3, 2], ['N', 1, 3], ['S', 6, 1], ['S', 4, 2], ['S', 5, 3]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count how many rows rank higher, using a correlated subquery.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`ROW_NUMBER() OVER (PARTITION BY ...)`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Count how many rows rank higher, using a correlated subquery.

```python
def numbered():
    return run_sql("""SELECT region, id, (SELECT COUNT(*) FROM sales b WHERE b.region = a.region AND
        (b.amount > a.amount OR (b.amount = a.amount AND b.id <= a.id))) AS rn
        FROM sales a ORDER BY region, rn""")
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `ROW_NUMBER() OVER (PARTITION BY ...)`.

```python
def numbered():
    return run_sql("""SELECT region, id, ROW_NUMBER() OVER (PARTITION BY region ORDER BY amount DESC, id) AS rn
        FROM sales ORDER BY region, rn""")
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count how many rows rank higher, using a correlated subquery. That costs O(n²) time and O(n) space."

"I can do better: `ROW_NUMBER() OVER (PARTITION BY ...)`. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/sql-advanced/number-the-rows)

<a id="q3"></a>
### Q3. Find Exact Duplicates

🟢 Warm-up · *Data quality*

Return (customer, email, updated) combinations appearing more than once.

**Examples**

- `` → `[[2, 'b@x', 15]]`

In [ ]:
def exact_dupes():
    """
    Return (customer, email, updated) combinations appearing more than once.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(exact_dupes, [
    ((), [[2, 'b@x', 15]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Self join on all the columns.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Group by all of them, with HAVING.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Self join on all the columns.

```python
def exact_dupes():
    return run_sql("""SELECT DISTINCT a.customer, a.email, a.updated FROM customer_updates a JOIN customer_updates b
        ON a.customer = b.customer AND a.email = b.email AND a.updated = b.updated AND a.id < b.id""")
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Group by all of them, with HAVING.

```python
def exact_dupes():
    return run_sql("SELECT customer, email, updated FROM customer_updates GROUP BY 1, 2, 3 HAVING COUNT(*) > 1")
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Self join on all the columns. That costs O(n²) time and O(n) space."

"I can do better: Group by all of them, with HAVING. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/sql-advanced/find-exact-duplicates)

<a id="q4"></a>
### Q4. RANK vs DENSE\_RANK

🟡 Interview standard · *Ties*

For region S ordered by amount descending, return id, RANK and DENSE\_RANK, ordered by rank, then id.

**Examples**

- `` → `[[6, 1, 1], [4, 2, 2], [5, 2, 2]]`

In [ ]:
def ranks():
    """
    For region S ordered by amount descending, return id, RANK and DENSE_RANK,
    ordered by rank, then id.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(ranks, [
    ((), [[6, 1, 1], [4, 2, 2], [5, 2, 2]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Counting subqueries.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Window functions. RANK leaves gaps after ties; DENSE\_RANK doesn't.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Counting subqueries.

```python
def ranks():
    return run_sql("""SELECT id,
        (SELECT COUNT(*) + 1 FROM sales b WHERE b.region = 'S' AND b.amount > a.amount) AS r,
        (SELECT COUNT(DISTINCT amount) + 1 FROM sales b WHERE b.region = 'S' AND b.amount > a.amount) AS d
        FROM sales a WHERE region = 'S' ORDER BY r, id""")
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Window functions. RANK leaves gaps after ties; DENSE\_RANK doesn't.

```python
def ranks():
    return run_sql("""SELECT id, RANK() OVER w, DENSE_RANK() OVER w FROM sales WHERE region = 'S'
        WINDOW w AS (ORDER BY amount DESC) ORDER BY 2, id""")
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Counting subqueries. That costs O(n²) time and O(n) space."

"I can do better: Window functions. RANK leaves gaps after ties; DENSE\_RANK doesn't. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/sql-advanced/rank-vs-dense-rank)

<a id="q5"></a>
### Q5. Top Sale per Region

🟡 Interview standard · *Top N per group*

Return region, id and amount of each region's largest sale.

**Examples**

- `` → `[['N', 2, 300], ['S', 6, 400]]`

In [ ]:
def top_sale():
    """
    Return region, id and amount of each region's largest sale.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(top_sale, [
    ((), [['N', 2, 300], ['S', 6, 400]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Join to a grouped MAX subquery.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

ROW\_NUMBER in a CTE, keeping number 1. Change 1 to any N.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Join to a grouped MAX subquery.

```python
def top_sale():
    return run_sql("""SELECT s.region, s.id, s.amount FROM sales s
        JOIN (SELECT region, MAX(amount) AS m FROM sales GROUP BY region) t
        ON t.region = s.region AND t.m = s.amount ORDER BY s.region""")
```

**Cost:** O(n) time, O(r) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** ROW\_NUMBER in a CTE, keeping number 1. Change 1 to any N.

```python
def top_sale():
    return run_sql("""WITH r AS (SELECT *, ROW_NUMBER() OVER (PARTITION BY region ORDER BY amount DESC) AS rn FROM sales)
        SELECT region, id, amount FROM r WHERE rn = 1 ORDER BY region""")
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Join to a grouped MAX subquery. That costs O(n) time and O(r) space."

"I can do better: ROW\_NUMBER in a CTE, keeping number 1. Change 1 to any N. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/sql-advanced/top-sale-per-region)

<a id="q6"></a>
### Q6. Running Total

🟡 Interview standard · *SUM OVER*

Return day, amount and running total within region N, by day.

**Examples**

- `` → `[['2026-01-01', 100, 100], ['2026-01-02', 300, 400], ['2026-01-03', 200, 600]]`

In [ ]:
def running_total():
    """
    Return day, amount and running total within region N, by day.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(running_total, [
    ((), [['2026-01-01', 100, 100], ['2026-01-02', 300, 400], ['2026-01-03', 200, 600]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A subquery summing all earlier rows.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`SUM(...) OVER (ORDER BY day)`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A subquery summing all earlier rows.

```python
def running_total():
    return run_sql("""SELECT day, amount, (SELECT SUM(amount) FROM sales b WHERE b.region = 'N' AND b.day <= a.day)
        FROM sales a WHERE region = 'N' ORDER BY day""")
```

**Cost:** O(n²) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `SUM(...) OVER (ORDER BY day)`.

```python
def running_total():
    return run_sql("""SELECT day, amount, SUM(amount) OVER (ORDER BY day ROWS UNBOUNDED PRECEDING)
        FROM sales WHERE region = 'N' ORDER BY day""")
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A subquery summing all earlier rows. That costs O(n²) time and O(1) space."

"I can do better: `SUM(...) OVER (ORDER BY day)`. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/sql-advanced/running-total)

<a id="q7"></a>
### Q7. Moving Average

🟡 Interview standard · *Frames*

For region S ordered by day, return id and the average of this row and the one before it, as a float.

**Examples**

- `` → `[[4, 50.0], [5, 50.0], [6, 225.0]]`

In [ ]:
def moving_average():
    """
    For region S ordered by day, return id and the average of this row and the
    one before it, as a float.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(moving_average, [
    ((), [[4, 50.0], [5, 50.0], [6, 225.0]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Self join to the previous row with ROW\_NUMBER.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A frame: `ROWS 1 PRECEDING`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Self join to the previous row with ROW\_NUMBER.

```python
def moving_average():
    return run_sql("""WITH t AS (SELECT id, amount, ROW_NUMBER() OVER (ORDER BY day) AS rn FROM sales WHERE region = 'S')
        SELECT a.id, CASE WHEN b.id IS NULL THEN a.amount * 1.0 ELSE (a.amount + b.amount) / 2.0 END
        FROM t a LEFT JOIN t b ON b.rn = a.rn - 1 ORDER BY a.rn""")
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A frame: `ROWS 1 PRECEDING`.

```python
def moving_average():
    return run_sql("""SELECT id, AVG(amount) OVER (ORDER BY day ROWS 1 PRECEDING) FROM sales
        WHERE region = 'S' ORDER BY day""")
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Self join to the previous row with ROW\_NUMBER. That costs O(n log n) time and O(n) space."

"I can do better: A frame: `ROWS 1 PRECEDING`. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/sql-advanced/moving-average)

<a id="q8"></a>
### Q8. Compare With the Previous Row

🟡 Interview standard · *LAG*

For region N by day, return id and change from the previous sale (NULL for the first).

**Examples**

- `` → `[[1, None], [2, 200], [3, -100]]`

In [ ]:
def change_from_previous():
    """
    For region N by day, return id and change from the previous sale (NULL for
    the first).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(change_from_previous, [
    ((), [[1, None], [2, 200], [3, -100]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A correlated subquery for the previous day's row.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`LAG`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A correlated subquery for the previous day's row.

```python
def change_from_previous():
    return run_sql("""SELECT id, amount - (SELECT b.amount FROM sales b WHERE b.region = 'N' AND b.day < a.day
        ORDER BY b.day DESC LIMIT 1) FROM sales a WHERE region = 'N' ORDER BY day""")
```

**Cost:** O(n²) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `LAG`.

```python
def change_from_previous():
    return run_sql("""SELECT id, amount - LAG(amount) OVER (ORDER BY day) FROM sales WHERE region = 'N' ORDER BY day""")
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A correlated subquery for the previous day's row. That costs O(n²) time and O(1) space."

"I can do better: `LAG`. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/sql-advanced/compare-with-the-previous-row)

<a id="q9"></a>
### Q9. Share Within a Group

🟡 Interview standard · *Window without collapsing*

Return id and its percentage of its region's total, rounded to 1 decimal, ordered by id.

**Examples**

- `` → `[[1, 16.7], [2, 50.0], [3, 33.3], [4, 10.0], [5, 10.0], [6, 80.0]]`

In [ ]:
def share_of_region():
    """
    Return id and its percentage of its region's total, rounded to 1 decimal,
    ordered by id.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(share_of_region, [
    ((), [[1, 16.7], [2, 50.0], [3, 33.3], [4, 10.0], [5, 10.0], [6, 80.0]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Join to grouped totals.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`SUM(...) OVER (PARTITION BY ...)` keeps every row.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Join to grouped totals.

```python
def share_of_region():
    return run_sql("""SELECT s.id, ROUND(100.0 * s.amount / t.total, 1) FROM sales s
        JOIN (SELECT region, SUM(amount) AS total FROM sales GROUP BY region) t ON t.region = s.region ORDER BY s.id""")
```

**Cost:** O(n) time, O(r) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `SUM(...) OVER (PARTITION BY ...)` keeps every row.

```python
def share_of_region():
    return run_sql("""SELECT id, ROUND(100.0 * amount / SUM(amount) OVER (PARTITION BY region), 1) FROM sales ORDER BY id""")
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Join to grouped totals. That costs O(n) time and O(r) space."

"I can do better: `SUM(...) OVER (PARTITION BY ...)` keeps every row. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/sql-advanced/share-within-a-group)

<a id="q10"></a>
### Q10. Latest Row per Key

🟡 Interview standard · *The most-used dedupe*

From customer\_updates, return customer and email of the newest row per customer (ties: highest id), ordered by customer.

**Examples**

- `` → `[[1, 'a2@x'], [2, 'b@x'], [3, 'c@x']]`

In [ ]:
def latest_row():
    """
    From customer_updates, return customer and email of the newest row per
    customer (ties: highest id), ordered by customer.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(latest_row, [
    ((), [[1, 'a2@x'], [2, 'b@x'], [3, 'c@x']]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

NOT EXISTS a newer row.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

ROW\_NUMBER ordered newest first, keeping 1.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** NOT EXISTS a newer row.

```python
def latest_row():
    return run_sql("""SELECT customer, email FROM customer_updates a WHERE NOT EXISTS (
        SELECT 1 FROM customer_updates b WHERE b.customer = a.customer
        AND (b.updated > a.updated OR (b.updated = a.updated AND b.id > a.id))) ORDER BY customer""")
```

**Cost:** O(n²) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** ROW\_NUMBER ordered newest first, keeping 1.

```python
def latest_row():
    return run_sql("""WITH r AS (SELECT *, ROW_NUMBER() OVER (PARTITION BY customer ORDER BY updated DESC, id DESC) AS rn
        FROM customer_updates) SELECT customer, email FROM r WHERE rn = 1 ORDER BY customer""")
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: NOT EXISTS a newer row. That costs O(n²) time and O(1) space."

"I can do better: ROW\_NUMBER ordered newest first, keeping 1. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/sql-advanced/latest-row-per-key)

<a id="q11"></a>
### Q11. Find Gaps in a Sequence

🟡 Interview standard · *Missing ranges*

From seats, return the start and end of each run of free seats (taken = 0), in order.

**Examples**

- `` → `[[3, 5], [7, 8]]`

In [ ]:
def free_runs():
    """
    From seats, return the start and end of each run of free seats (taken = 0),
    in order.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(free_runs, [
    ((), [[3, 5], [7, 8]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Walk the rows in Python.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Islands: seat − ROW\_NUMBER among free seats.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Walk the rows in Python.

```python
def free_runs():
    out = []
    for seat, taken in run_sql("SELECT seat, taken FROM seats ORDER BY seat"):
        if taken:
            continue
        if out and out[-1][1] == seat - 1:
            out[-1][1] = seat
        else:
            out.append([seat, seat])
    return out
```

**Cost:** O(n) time, O(g) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Islands: seat − ROW\_NUMBER among free seats.

```python
def free_runs():
    return run_sql("""WITH f AS (SELECT seat, seat - ROW_NUMBER() OVER (ORDER BY seat) AS grp FROM seats WHERE taken = 0)
        SELECT MIN(seat), MAX(seat) FROM f GROUP BY grp ORDER BY 1""")
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Walk the rows in Python. That costs O(n) time and O(g) space."

"I can do better: Islands: seat − ROW\_NUMBER among free seats. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/sql-advanced/find-gaps-in-a-sequence)

<a id="q12"></a>
### Q12. Funnel Conversion

🟡 Interview standard · *Who reached each step?*

Return distinct users who viewed 'home', and of those how many reached 'cart', then 'buy', as one row.

**Examples**

- `` → `[2, 1, 1]`

In [ ]:
def funnel():
    """
    Return distinct users who viewed 'home', and of those how many reached
    'cart', then 'buy', as one row.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(funnel, [
    ((), [2, 1, 1]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Three queries, using sets in Python.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Conditional aggregation per user, then sum.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Three queries, using sets in Python.

```python
def funnel():
    pages = {}
    for u, p in run_sql("SELECT user_id, page FROM events"):
        pages.setdefault(p, set()).add(u)
    home = pages.get("home", set())
    cart = home & pages.get("cart", set())
    return [len(home), len(cart), len(cart & pages.get("buy", set()))]
```

**Cost:** O(n) time, O(u) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Conditional aggregation per user, then sum.

```python
def funnel():
    return run_sql("""WITH f AS (SELECT user_id, MAX(page = 'home') AS h, MAX(page = 'cart') AS c, MAX(page = 'buy') AS b
        FROM events GROUP BY user_id)
        SELECT SUM(h), SUM(h AND c), SUM(h AND c AND b) FROM f""")[0]
```

**Cost:** O(n) time, O(u) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Three queries, using sets in Python. That costs O(n) time and O(u) space."

"I can do better: Conditional aggregation per user, then sum. That's O(n) time and O(u) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/sql-advanced/funnel-conversion)

<a id="q13"></a>
### Q13. Pivot Rows to Columns

🟡 Interview standard · *Reports*

For each rep, return amount in region N and region S as separate columns, ordered by rep.

**Examples**

- `` → `[['a', 300, 0], ['b', 300, 0], ['c', 0, 450], ['d', 0, 50]]`

In [ ]:
def pivot_regions():
    """
    For each rep, return amount in region N and region S as separate columns,
    ordered by rep.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(pivot_regions, [
    ((), [['a', 300, 0], ['b', 300, 0], ['c', 0, 450], ['d', 0, 50]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Two scalar subqueries per rep.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Conditional aggregation.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Two scalar subqueries per rep.

```python
def pivot_regions():
    return run_sql("""SELECT DISTINCT rep,
        (SELECT COALESCE(SUM(amount),0) FROM sales b WHERE b.rep = a.rep AND b.region = 'N'),
        (SELECT COALESCE(SUM(amount),0) FROM sales b WHERE b.rep = a.rep AND b.region = 'S') FROM sales a ORDER BY rep""")
```

**Cost:** O(r·n) time, O(r) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Conditional aggregation.

```python
def pivot_regions():
    return run_sql("""SELECT rep, SUM(CASE WHEN region = 'N' THEN amount ELSE 0 END),
        SUM(CASE WHEN region = 'S' THEN amount ELSE 0 END) FROM sales GROUP BY rep ORDER BY rep""")
```

**Cost:** O(n) time, O(r) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Two scalar subqueries per rep. That costs O(r·n) time and O(r) space."

"I can do better: Conditional aggregation. That's O(n) time and O(r) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/sql-advanced/pivot-rows-to-columns)

<a id="q14"></a>
### Q14. Generate a Date Spine

🟡 Interview standard · *Fill the missing days*

Return every day from 2026-01-01 to 2026-01-04 with region S's total, showing 0 for days without sales.

**Examples**

- `` → `[['2026-01-01', 50], ['2026-01-02', 50], ['2026-01-03', 0], ['2026-01-04', 400]]`

In [ ]:
def daily_with_zeros():
    """
    Return every day from 2026-01-01 to 2026-01-04 with region S's total,
    showing 0 for days without sales.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(daily_with_zeros, [
    ((), [['2026-01-01', 50], ['2026-01-02', 50], ['2026-01-03', 0], ['2026-01-04', 400]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Fill in missing days in Python.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A recursive CTE of dates, LEFT JOINed to the sales.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Fill in missing days in Python.

```python
def daily_with_zeros():
    have = dict(map(tuple, run_sql("SELECT day, SUM(amount) FROM sales WHERE region = 'S' GROUP BY day")))
    return [[d, have.get(d, 0)] for d in ["2026-01-01", "2026-01-02", "2026-01-03", "2026-01-04"]]
```

**Cost:** O(d) time, O(d) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A recursive CTE of dates, LEFT JOINed to the sales.

```python
def daily_with_zeros():
    return run_sql("""WITH RECURSIVE d(day) AS (SELECT '2026-01-01' UNION ALL
            SELECT date(day, '+1 day') FROM d WHERE day < '2026-01-04')
        SELECT d.day, COALESCE(SUM(s.amount), 0) FROM d LEFT JOIN sales s ON s.day = d.day AND s.region = 'S'
        GROUP BY d.day ORDER BY d.day""")
```

**Cost:** O(d + n) time, O(d) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Fill in missing days in Python. That costs O(d) time and O(d) space."

"I can do better: A recursive CTE of dates, LEFT JOINed to the sales. That's O(d + n) time and O(d) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/sql-advanced/generate-a-date-spine)

<a id="q15"></a>
### Q15. First and Last in a Group

🟡 Interview standard · *FIRST\_VALUE*

For each user in events, return the first and last page visited, ordered by user.

**Examples**

- `` → `[[1, 'home', 'buy'], [2, 'home', 'home'], [3, 'cart', 'cart']]`

In [ ]:
def first_last_page():
    """
    For each user in events, return the first and last page visited, ordered by
    user.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(first_last_page, [
    ((), [[1, 'home', 'buy'], [2, 'home', 'home'], [3, 'cart', 'cart']]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Two ROW\_NUMBER lookups joined together.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`FIRST_VALUE`, with a full frame for the last.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Two ROW\_NUMBER lookups joined together.

```python
def first_last_page():
    rows = run_sql("SELECT user_id, page FROM events ORDER BY user_id, ts")
    first, last = {}, {}
    for u, p in rows:
        first.setdefault(u, p)
        last[u] = p
    return [[u, first[u], last[u]] for u in sorted(first)]
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `FIRST_VALUE`, with a full frame for the last.

```python
def first_last_page():
    return run_sql("""SELECT DISTINCT user_id,
        FIRST_VALUE(page) OVER (PARTITION BY user_id ORDER BY ts),
        FIRST_VALUE(page) OVER (PARTITION BY user_id ORDER BY ts DESC) FROM events ORDER BY user_id""")
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Two ROW\_NUMBER lookups joined together. That costs O(n log n) time and O(n) space."

"I can do better: `FIRST_VALUE`, with a full frame for the last. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/sql-advanced/first-and-last-in-a-group)

<a id="q16"></a>
### Q16. Split Into Buckets

🟡 Interview standard · *NTILE*

Split all sales by amount into 3 equal buckets (ties by id), returning id and bucket, ordered by id.

**Examples**

- `` → `[[1, 2], [2, 3], [3, 2], [4, 1], [5, 1], [6, 3]]`

In [ ]:
def buckets():
    """
    Split all sales by amount into 3 equal buckets (ties by id), returning id
    and bucket, ordered by id.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(buckets, [
    ((), [[1, 2], [2, 3], [3, 2], [4, 1], [5, 1], [6, 3]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Number the rows, then compute the bucket.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`NTILE(3)`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Number the rows, then compute the bucket.

```python
def buckets():
    rows = run_sql("SELECT id, ROW_NUMBER() OVER (ORDER BY amount, id) FROM sales")
    n = len(rows)
    return sorted([[i, (r - 1) * 3 // n + 1] for i, r in rows])
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `NTILE(3)`.

```python
def buckets():
    return run_sql("SELECT id, NTILE(3) OVER (ORDER BY amount, id) FROM sales ORDER BY id")
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Number the rows, then compute the bucket. That costs O(n log n) time and O(n) space."

"I can do better: `NTILE(3)`. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/sql-advanced/split-into-buckets)

<a id="q17"></a>
### Q17. Sessionise Clicks

🔴 Stretch · *New session after 30 minutes*

Give each event a session number per user, where a gap above 1800 seconds starts a new session. Return user\_id, ts and session, ordered by user and time.

**Examples**

- `` → `[[1, 100, 1], [1, 160, 1], [1, 2000, 2], [1, 2050, 2], [2, 300, 1], [2, 4000, 2], [3, 500, 1]]`

In [ ]:
def sessions():
    """
    Give each event a session number per user, where a gap above 1800 seconds
    starts a new session. Return user_id, ts and session, ordered by user and
    time.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(sessions, [
    ((), [[1, 100, 1], [1, 160, 1], [1, 2000, 2], [1, 2050, 2], [2, 300, 1], [2, 4000, 2], [3, 500, 1]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Fetch rows and walk them in Python.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

LAG finds gaps; a running SUM of 'is new session' numbers them.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Fetch rows and walk them in Python.

```python
def sessions():
    out, last_user, last_ts, session = [], None, None, 0
    for user, ts in run_sql("SELECT user_id, ts FROM events ORDER BY user_id, ts"):
        if user != last_user:
            session = 1
        elif ts - last_ts > 1800:
            session += 1
        out.append([user, ts, session])
        last_user, last_ts = user, ts
    return out
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** LAG finds gaps; a running SUM of 'is new session' numbers them.

```python
def sessions():
    return run_sql("""WITH g AS (
            SELECT user_id, ts, CASE WHEN ts - LAG(ts) OVER w > 1800 THEN 1 ELSE 0 END AS new
            FROM events WINDOW w AS (PARTITION BY user_id ORDER BY ts))
        SELECT user_id, ts, 1 + SUM(new) OVER (PARTITION BY user_id ORDER BY ts) FROM g ORDER BY user_id, ts""")
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Fetch rows and walk them in Python. That costs O(n log n) time and O(n) space."

"I can do better: LAG finds gaps; a running SUM of 'is new session' numbers them. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/sql-advanced/sessionise-clicks)

<a id="q18"></a>
### Q18. Longest Login Streak

🔴 Stretch · *Gaps and islands*

Return each user's longest run of consecutive days, ordered by user.

**Examples**

- `` → `[[1, 3], [2, 1], [3, 2]]`

In [ ]:
def longest_streak():
    """
    Return each user's longest run of consecutive days, ordered by user.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(longest_streak, [
    ((), [[1, 3], [2, 1], [3, 2]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Walk the rows in Python.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Day minus ROW\_NUMBER stays constant within a streak.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Walk the rows in Python.

```python
def longest_streak():
    best, prev_user, prev_day, run = {}, None, None, 0
    for user, day in run_sql("SELECT user_id, day FROM logins ORDER BY user_id, day"):
        run = run + 1 if user == prev_user and day == prev_day + 1 else 1
        best[user] = max(best.get(user, 0), run)
        prev_user, prev_day = user, day
    return [[u, best[u]] for u in sorted(best)]
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Day minus ROW\_NUMBER stays constant within a streak.

```python
def longest_streak():
    return run_sql("""WITH g AS (SELECT user_id, day - ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY day) AS grp FROM logins),
        s AS (SELECT user_id, COUNT(*) AS n FROM g GROUP BY user_id, grp)
        SELECT user_id, MAX(n) FROM s GROUP BY user_id ORDER BY user_id""")
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Walk the rows in Python. That costs O(n log n) time and O(n) space."

"I can do better: Day minus ROW\_NUMBER stays constant within a streak. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/sql-advanced/longest-login-streak)

<a id="q19"></a>
### Q19. Weekly Retention

🔴 Stretch · *Cohorts*

For each signup week and weeks since signup, return the number of active users. Order by cohort, then offset.

**Examples**

- `` → `[[0, 0, 3], [0, 1, 1], [0, 2, 1], [1, 0, 2], [1, 1, 1]]`

In [ ]:
def retention():
    """
    For each signup week and weeks since signup, return the number of active
    users. Order by cohort, then offset.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(retention, [
    ((), [[0, 0, 3], [0, 1, 1], [0, 2, 1], [1, 0, 2], [1, 1, 1]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Dictionaries of sets in Python.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Join activity to signups, and group by cohort and offset.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Dictionaries of sets in Python.

```python
def retention():
    signup = dict(map(tuple, run_sql("SELECT user_id, week FROM signups")))
    groups = {}
    for user, week in run_sql("SELECT user_id, week FROM activity"):
        groups.setdefault((signup[user], week - signup[user]), set()).add(user)
    return [[c, o, len(groups[(c, o)])] for c, o in sorted(groups)]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Join activity to signups, and group by cohort and offset.

```python
def retention():
    return run_sql("""SELECT s.week, a.week - s.week AS offset, COUNT(DISTINCT a.user_id)
        FROM activity a JOIN signups s ON s.user_id = a.user_id GROUP BY 1, 2 ORDER BY 1, 2""")
```

**Cost:** O(n) time, O(c·w) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Dictionaries of sets in Python. That costs O(n) time and O(n) space."

"I can do better: Join activity to signups, and group by cohort and offset. That's O(n) time and O(c·w) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/sql-advanced/weekly-retention)

<a id="q20"></a>
### Q20. Walk a Hierarchy

🔴 Stretch · *Recursive CTE*

Return each category's name and depth (root = 0), ordered by depth, then name.

**Examples**

- `` → `[['all', 0], ['food', 1], ['home', 1], ['tea', 2], ['green', 3]]`

In [ ]:
def depths():
    """
    Return each category's name and depth (root = 0), ordered by depth, then
    name.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(depths, [
    ((), [['all', 0], ['food', 1], ['home', 1], ['tea', 2], ['green', 3]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Search level by level in Python.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A recursive CTE.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Search level by level in Python.

```python
def depths():
    rows = run_sql("SELECT id, parent, name FROM tree")
    out, level, frontier = [], 0, [r for r in rows if r[1] is None]
    while frontier:
        out += [[r[2], level] for r in frontier]
        ids = {r[0] for r in frontier}
        frontier = [r for r in rows if r[1] in ids]
        level += 1
    return sorted(out, key=lambda x: (x[1], x[0]))
```

**Cost:** O(n·d) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A recursive CTE.

```python
def depths():
    return run_sql("""WITH RECURSIVE t(id, name, d) AS (
            SELECT id, name, 0 FROM tree WHERE parent IS NULL
            UNION ALL SELECT c.id, c.name, t.d + 1 FROM tree c JOIN t ON c.parent = t.id)
        SELECT name, d FROM t ORDER BY d, name""")
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Search level by level in Python. That costs O(n·d) time and O(n) space."

"I can do better: A recursive CTE. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/sql-advanced/walk-a-hierarchy)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. ROW\_NUMBER vs RANK vs DENSE\_RANK?</b></summary>

On ties, ROW\_NUMBER still gives different numbers, RANK shares and leaves a gap, and DENSE\_RANK shares with no gap.

</details>

<details>
<summary><b>2. GROUP BY vs window function?</b></summary>

GROUP BY collapses rows to one per group. A window computes across rows while keeping them all.

</details>

<details>
<summary><b>3. ROWS vs RANGE in a frame?</b></summary>

ROWS counts physical rows. RANGE groups rows with equal ORDER BY values, so ties share a result. The default with ORDER BY is usually RANGE.

</details>

<details>
<summary><b>4. Why can't you use a window function in WHERE?</b></summary>

WHERE runs before windows are computed. Wrap the query in a CTE or subquery, and filter outside it.

</details>

<details>
<summary><b>5. CTE vs subquery vs temp table?</b></summary>

CTEs and subqueries are logical, and CTEs read better. A temp table actually stores results, which helps when a step's used several times or the optimiser needs real statistics.

</details>

<details>
<summary><b>6. Scenario: a dedupe query gives different results on each run. Why?</b></summary>

The ROW\_NUMBER ordering has ties, so which row wins is arbitrary. Add a deterministic tiebreaker, such as an id or ingestion time.

</details>

<details>
<summary><b>7. Scenario: a daily report misses days with no sales. How do you fix it?</b></summary>

Build a date spine, using a calendar table or generated series, and LEFT JOIN the facts, using COALESCE to show 0.

</details>

<details>
<summary><b>8. Scenario: a window query is very slow on a huge table. What do you try?</b></summary>

Filter first, reduce the columns, make sure partition sizes are sensible, and check for skew. Possibly pre-aggregate. Windows need sorting, which is costly at scale.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: CTEs: write SQL you can read
- [ ] I can explain: Window functions: the big idea
- [ ] I can explain: Running totals, moving averages and LAG
- [ ] I can explain: Sessions, streaks and gaps
- [ ] I can explain: Funnels, retention, pivots and hierarchies

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Split Into Buckets
- [ ] [Q17](#q17) Sessionise Clicks
- [ ] [Q18](#q18) Longest Login Streak
- [ ] [Q19](#q19) Weekly Retention
- [ ] [Q20](#q20) Walk a Hierarchy

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.